In [ ]:
# =============================================================
# Install + Imports
# =============================================================
!pip install "lerobot @ git+https://github.com/huggingface/lerobot.git" -q
!pip install num2words -q

import torch
import json
from collections import defaultdict
print("Imports done.")

In [ ]:
# =============================================================
# Load SmolVLA on CPU
# =============================================================
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")
policy = policy.to("cpu").eval()
print(f"Loaded: {type(policy).__name__}")

In [ ]:
# =============================================================
# Inspecting the modules
# =============================================================
for name, module in policy.named_children():
  print(name)
  print(f"-----------------------")
  print(module)



In [ ]:
# =============================================================
# Architecture Tree (module | params | trainable?)
# =============================================================
def print_architecture(model, prefix="", max_depth=4):
    for name, module in model.named_children():
        full_name = f"{prefix}.{name}" if prefix else name
        total_params = sum(p.numel() for p in module.parameters())
        total_trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        depth = full_name.count(".")
        indent = "  " * depth
        if total_params > 0:
            if total_trainable == 0:
                tag = "FROZEN"
            elif total_trainable == total_params:
                tag = "TRAINABLE"
            else:
                tag = f"MIXED: {total_trainable/total_params*100:.1f}%"
            print(f"{indent}{name}: {type(module).__name__} | {total_params:,} params [{tag}]")
        else:
            print(f"{indent}{name}: {type(module).__name__}")
        if depth < max_depth:
            print_architecture(module, full_name, max_depth)

print_architecture(policy, max_depth=3)

In [ ]:
# =============================================================
# Frozen vs Trainable Breakdown
# =============================================================
total_params = 0
trainable_params = 0
frozen_params = 0
module_stats = defaultdict(lambda: {"total": 0, "trainable": 0, "frozen": 0})

for name, param in policy.named_parameters():
    n = param.numel()
    total_params += n
    parts = name.split(".")
    key = f"{parts[0]}.{parts[1]}" if len(parts) > 1 else parts[0]
    module_stats[key]["total"] += n
    if param.requires_grad:
        trainable_params += n
        module_stats[key]["trainable"] += n
    else:
        frozen_params += n
        module_stats[key]["frozen"] += n

print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")
print(f"Frozen parameters:    {frozen_params:>12,}")
print(f"Trainable percentage: {trainable_params/total_params*100:.2f}%")
print(f"Model size (fp32):    {total_params * 4 / 1024**2:.1f} MB")
print(f"Model size (fp16):    {total_params * 2 / 1024**2:.1f} MB")
print(f"\n{'Module':<45} {'Total':>12} {'Trainable':>12} {'Frozen':>12} {'%Train':>8}")
print("-" * 91)
for key in sorted(module_stats.keys()):
    s = module_stats[key]
    pct = s["trainable"] / s["total"] * 100 if s["total"] > 0 else 0
    print(f"{key:<45} {s['total']:>12,} {s['trainable']:>12,} {s['frozen']:>12,} {pct:>7.1f}%")

In [ ]:
# =============================================================
# Action Expert Deep Inspection
# =============================================================
print("--- All named modules with own params > 0 ---\n")
for name, module in policy.named_modules():
    params = sum(p.numel() for p in module.parameters(recurse=False))
    if params > 0:
        trainable = sum(p.numel() for p in module.parameters(recurse=False) if p.requires_grad)
        tag = "TRAIN" if trainable > 0 else "FROZEN"
        print(f"  {name}: {type(module).__name__} ({params:,} own params) [{tag}]")

In [ ]:
# =============================================================
# Attention Layers (Cross-Attn + Self-Attn Q/K/V dims)
# =============================================================
for name, module in policy.named_modules():
    mod_type = type(module).__name__
    if "attention" in mod_type.lower() or "attn" in name.lower():
        print(f"\n{name}: {mod_type}")
        for pname, param in module.named_parameters(recurse=False):
            print(f"  {pname}: shape={list(param.shape)}, requires_grad={param.requires_grad}")

In [ ]:
# =============================================================
# Flow Matching / Time Embedding / Output Head
# =============================================================
for name, module in policy.named_modules():
    name_lower = name.lower()
    mod_type = type(module).__name__.lower()
    if any(kw in name_lower or kw in mod_type
           for kw in ["flow", "time", "timestep", "noise", "denois", "velocity", "head"]):
        params = sum(p.numel() for p in module.parameters(recurse=False))
        if params > 0:
            print(f"\n{name}: {type(module).__name__} ({params:,} params)")
            for pname, param in module.named_parameters(recurse=False):
                print(f"  {pname}: shape={list(param.shape)}")

In [ ]:
# =============================================================
# VLM Backbone (SigLIP + SmolLM2)
# =============================================================
for name, module in policy.named_modules():
    name_lower = name.lower()
    if any(kw in name_lower for kw in ["siglip", "vision", "vlm", "smollm", "language"]):
        params = sum(p.numel() for p in module.parameters())
        if params > 1000:
            trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
            status = "FROZEN" if trainable == 0 else f"{trainable/params*100:.1f}% trainable"
            depth = name.count(".")
            if depth <= 3:
                print(f"  {name}: {type(module).__name__} ({params:,} params) [{status}]")

In [ ]:
# =============================================================
# Policy Config Dump
# =============================================================
config = policy.config
print(f"Config type: {type(config).__name__}\n")
for attr in sorted(dir(config)):
    if not attr.startswith("_"):
        val = getattr(config, attr, None)
        if not callable(val):
            print(f"  {attr}: {val}")

In [ ]:
# =============================================================
# LayerNorm Params
# =============================================================
layernorm_params = 0
layernorm_trainable = 0

for name, param in policy.named_parameters():
    if any(kw in name.lower() for kw in ["norm", "layernorm", "ln"]):
        layernorm_params += param.numel()
        if param.requires_grad:
            layernorm_trainable += param.numel()
        print(f"  {name}: shape={list(param.shape)}, requires_grad={param.requires_grad}")

print(f"\nLayerNorm summary:")
print(f"  Total LayerNorm params:       {layernorm_params:,}")
print(f"  Currently trainable:          {layernorm_trainable:,}")
print(f"  As % of all trainable params: {layernorm_params/trainable_params*100:.3f}%")
print(f"  As % of all params:           {layernorm_params/total_params*100:.3f}%")
